# 05-06 Configure: 런타임에 체인 설정 바꾸기

## 학습 목표

이 노트북을 완료하면 다음을 할 수 있어요:

- `configurable_fields`로 체인 실행 시점에 모델 이름, temperature 등 파라미터를 동적으로 변경하는 방법을 구현할 수 있어요
- `configurable_alternatives`로 런타임에 완전히 다른 모델이나 프롬프트를 교체하는 방법을 설명할 수 있어요
- `with_config()`로 특정 설정을 미리 적용한 체인 버전을 만들고 재사용할 수 있어요

## 사전 지식

이 노트북을 시작하기 전에 다음 개념을 알고 있으면 좋아요:

- `ChatOpenAI` 모델 초기화와 파라미터 설정 방법
- LCEL 파이프 연산자(`|`) 사용법
- 파이썬 딕셔너리와 키워드 인자 전달 방법

---

체인을 호출할 때 다양한 옵션을 동적으로 설정하는 방법을 배워요.

**두 가지 방식:**

1. **`configurable_fields`**: 실행 가능한 객체의 특정 필드를 동적으로 구성
   - 예: 모델명, temperature 등 파라미터를 런타임에 변경

2. **`configurable_alternatives`**: 특정 실행 가능한 객체에 대한 대안을 런타임에 선택
   - 예: 여러 모델 중 선택, 여러 프롬프트 중 선택

이를 통해 하나의 체인으로 다양한 설정을 유연하게 관리할 수 있어요.

In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()


True

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import ConfigurableField
from langchain_openai import ChatOpenAI

# 기본 모델 초기화
model = ChatOpenAI(temperature=0, model_name="gpt-4o-mini")


## 1. configurable_fields — 파라미터를 동적으로 변경하기

`configurable_fields`를 사용하면 모델의 특정 필드를 런타임에 동적으로 변경할 수 있어요.

예를 들어, `model_name` 파라미터를 동적으로 설정하여 호출 시점에 다른 모델을 사용할 수 있어요.

설정 방법은 두 가지가 있어요:

```python
# 방법 1: invoke() 시 매번 config 전달
model.invoke("질문", config={"configurable": {"gpt_version": "gpt-4o-mini"}})

# 방법 2: with_config()로 설정을 미리 고정 후 재사용
gpt4o_model = model.with_config(configurable={"gpt_version": "gpt-4o-mini"})
gpt4o_model.invoke("질문")
```

> **실무 팁:** A/B 테스트나 사용자 요금제별로 다른 모델을 적용할 때 유용해요. 하나의 체인 코드베이스에서 `config` 값만 바꿔 여러 모델을 관리할 수 있어요.

In [5]:
# ============================================================
# TODO: configurable_fields()로 model_name을 동적으로 변경 가능하도록 설정하세요
# 힌트: ChatOpenAI(...).configurable_fields(model_name=ConfigurableField(id="gpt_version", ...))
# 예상 결과: invoke() 시 config={"configurable": {"gpt_version": "..."}}으로 모델 교체
# ============================================================

# ---------------------------------------------------
# configurable_fields: 파라미터 값을 런타임에 동적으로 변경
# ---------------------------------------------------
configurable_model = ChatOpenAI(temperature=0).configurable_fields(
    model_name=ConfigurableField(
        id="gpt_version",
        name="Version of GPT",
        description="사용할 GPT 모델 버전. 예) gpt-4o"
    ),
)

In [ ]:
# 기본 모델로 실행 (설정 없이)
result_default = configurable_model.invoke("세종대왕의 맥북 던짐 사건 설명해줘")
print(f' ==> [Line 2]: \033[38;2;131;184;29m[result_default]\033[0m({type(result_default).__name__}) = \033[38;2;79;214;205m{result_default}\033[0m')


 ==> [Line 2]: [result_default](AIMessage) = content='세종대왕의 맥북 던짐 사건은 한국의 역사 속에서 유명한 사건 중 하나로, 조선 시대의 제4대 군주인 세종대왕이 맥북을 던진 사건을 가리킨다.\n\n세종대왕은 조선시대의 대표적인 왕으로, 한글을 창제하고 학문과 문화를 발전시킨 위인으로 알려져 있다. 그러나 그의 성격은 엄격하고 까다로웠던 것으로 전해진다.\n\n한 번 세종대왕이 학자들과 토론을 벌이던 중, 학자들이 세종대왕의 의견에 동의하지 않아 분노한 세종대왕은 자신의 맥북을 땅에 내리치며 "이것이 내 의견이다!"라고 외쳤다고 전해진다.\n\n이 사건은 세종대왕의 엄격한 성격과 까다로운 면모를 보여주는 에피소드로 전해지며, 세종대왕의 통치 스타일과 인격을 이해하는 데 도움을 준다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 370, 'prompt_tokens': 32, 'total_tokens': 402, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DOyIYXU4J9mgGIwR4LKrGqJz0Edms', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--019d3cf2-adcf-7700-99c4-459992f4887a-0' usage_metad

In [8]:
# config로 모델 변경하여 실행
# 
# config={"configurable": {"키": "값"}} 형식으로 동적 설정
result_gpt4o = configurable_model.invoke(
  "세종대왕의 맥북 던짐 사건 설명해줘",
  config={"configurable": {"gpt_version": "gpt-4o"}}
)

print(result_gpt4o)

content='세종대왕의 "맥북 던짐 사건"은 실제 역사적 사건이 아닙니다. 세종대왕은 조선 시대의 네 번째 왕으로, 15세기 초에 살았던 인물입니다. 따라서 현대의 기술인 맥북과 관련된 사건이 있을 수 없습니다. 이와 같은 이야기는 아마도 인터넷 밈이나 유머로 만들어진 가상의 이야기일 가능성이 높습니다. 세종대왕은 한글 창제와 같은 중요한 업적으로 잘 알려져 있으며, 그의 통치 기간 동안 많은 문화적, 과학적 발전이 이루어졌습니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 131, 'prompt_tokens': 22, 'total_tokens': 153, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_05f00232ad', 'id': 'chatcmpl-DOyMEmIjCkN4prdNzYqZJZGqvYKEj', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--019d3cf6-284b-78d0-9fd0-d9db577baee7-0' usage_metadata={'input_tokens': 22, 'output_tokens': 131, 'total_tokens': 153, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 

In [ ]:
# 다른 모델로 실행


## 섹션 전환 — with_config()와 달라지는 것

`configurable_fields`로 파라미터를 변경하는 방법을 살펴봤어요. 이번에는 모델이나 프롬프트 자체를 완전히 교체하는 `configurable_alternatives`를 알아볼게요.

In [9]:
# with_config()로 설정을 미리 지정
gpt4o_model = configurable_model.with_config(
    configurable={"gpt_version": "gpt-4o-mini"}
)

res = gpt4o_model.invoke("세종대왕의 맥북 던짐 사건 설명해줘")
res

AIMessage(content='"세종대왕의 맥북 던짐 사건"은 역사적 사실이 아닌 현대의 유머나 패러디로 보입니다. 세종대왕은 조선의 제4대 왕으로, 한글을 창제하고 과학, 문화, 정치 등 여러 분야에서 많은 업적을 남긴 인물입니다. 그러나 그가 맥북을 던졌다는 이야기는 역사적 사실과는 관련이 없습니다.\n\n이런 표현은 현대의 기술과 역사적 인물을 결합하여 유머를 만들거나, 특정 상황을 비유적으로 설명하기 위해 사용될 수 있습니다. 만약 이 사건에 대한 특정한 맥락이나 출처가 있다면, 더 구체적인 정보를 제공해 주시면 좋겠습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 159, 'prompt_tokens': 22, 'total_tokens': 181, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5714259854', 'id': 'chatcmpl-DOyNlga7abVcxzm6dsvkZaxEH8MUp', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019d3cf7-9afd-79e0-85c0-77836d1011a4-0', usage_metadata={'input_tokens': 22, 'output_tokens': 159, 'total_tokens': 181, 'input_token_details':

## 2. configurable_alternatives — Runnable 자체를 교체하기

`configurable_alternatives`를 사용하면 런타임에 여러 Runnable 중 하나를 선택할 수 있어요.

`configurable_fields`가 **값**을 바꾸는 것이라면, `configurable_alternatives`는 **객체** 자체를 바꾸는 것이에요.

> **주의:** `default_key`는 `configurable_alternatives`에서 반드시 지정해야 해요. 이 키가 설정 없이 체인을 실행할 때 사용되는 기본 Runnable을 결정해요.

In [11]:
# ============================================================
# TODO: configurable_alternatives()로 여러 모델을 런타임에 선택 가능하도록 설정하세요
# 힌트: ChatOpenAI(...).configurable_alternatives(ConfigurableField(id="llm"), default_key="openai_mini", openai=..., openai_turbo=...)
# 주의: default_key에 지정한 이름과 동일한 키워드 인자를 넣으면 중복 에러 발생!
# 예상 결과: with_config(configurable={"llm": "openai"})로 모델 교체
# ============================================================

# ---------------------------------------------------
# configurable_alternatives: Runnable 객체 자체를 런타임에 교체
# ---------------------------------------------------

llm = ChatOpenAI(temperature=0, model_name="gpt-4o-mini").configurable_alternatives(
    ConfigurableField(id="llm"),
    default_key="openai_mini",

    openai=ChatOpenAI(model_name="gpt-4o"),
    openai_turbo=ChatOpenAI(model_name="gpt-3.5-turbo")
)

prompt = ChatPromptTemplate.from_template("{topic}에 대해 간단히 설명하세요.")

chain = prompt | llm

In [12]:
# 기본 모델로 실행
res = chain.invoke({"topic": "블록체인"})
print(f' ==> [Line 2]: \033[38;2;117;234;35m[res]\033[0m({type(res).__name__}) = \033[38;2;206;209;30m{res}\033[0m')


 ==> [Line 2]: [res](AIMessage) = content="블록체인은 데이터를 안전하게 저장하고 관리하기 위한 분산 원장 기술입니다. 기본적으로 블록체인은 여러 개의 '블록'으로 구성되어 있으며, 각 블록은 거래 정보나 데이터를 포함하고 있습니다. 이 블록들은 시간 순서대로 연결되어 '체인'을 형성합니다.\n\n블록체인의 주요 특징은 다음과 같습니다:\n\n1. **분산성**: 블록체인은 중앙 서버가 아닌 여러 컴퓨터(노드)에 분산되어 저장됩니다. 이로 인해 데이터의 위변조가 어렵고, 시스템의 신뢰성이 높아집니다.\n\n2. **투명성**: 모든 거래 기록이 공개되어 누구나 확인할 수 있습니다. 이는 거래의 신뢰성을 높이는 데 기여합니다.\n\n3. **불변성**: 한 번 기록된 데이터는 수정하거나 삭제할 수 없기 때문에, 거래의 이력을 안전하게 보존할 수 있습니다.\n\n4. **암호화**: 블록체인은 강력한 암호화 기술을 사용하여 데이터의 안전성을 보장합니다.\n\n이러한 특성 덕분에 블록체인은 금융 거래, 스마트 계약, 공급망 관리 등 다양한 분야에서 활용되고 있습니다." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 258, 'prompt_tokens': 19, 'total_tokens': 277, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5714259854', 'id': 'chat

In [13]:
# 다른 모델로 실행
res = chain.with_config(configurable={"llm": "openai_turbo"}).invoke({"topic": "블록체인"}, )
res


AIMessage(content='블록체인은 분산 데이터 저장 기술로, 일련의 데이터 블록이 연결된 체인 형태로 구성되어 있습니다. 각 블록은 이전 블록의 정보와 암호화된 데이터를 포함하고 있어 데이터의 위변조를 방지할 수 있습니다. 블록체인은 중앙 기관의 개입 없이 안전하고 신뢰성 있는 거래를 가능하게 해주는 기술로 주로 가상화폐 거래에서 사용되지만 다양한 분야에서 활용되고 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 163, 'prompt_tokens': 26, 'total_tokens': 189, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DOyYzwEkg7RFQjVNxcjuwZUsXQrLv', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019d3d02-3967-7651-80c0-d8fc6d039259-0', usage_metadata={'input_tokens': 26, 'output_tokens': 163, 'total_tokens': 189, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## 3. 프롬프트 대안 설정

프롬프트도 `configurable_alternatives`를 사용하여 런타임에 선택할 수 있어요.

In [14]:
# ============================================================
# TODO: 프롬프트에도 configurable_alternatives를 설정하고 런타임에 교체하세요
# 힌트: ChatPromptTemplate.from_template(...).configurable_alternatives(ConfigurableField(id="prompt"), default_key="capital", area=..., population=...)
# 주의: default_key="capital"이므로 capital=... 키워드 인자는 넣지 마세요 (기본 Runnable이 자동 매핑됨)
# 예상 결과: with_config(configurable={"prompt": "area"})로 프롬프트 교체
# ============================================================

# ---------------------------------------------------
# 프롬프트에 configurable_alternatives 적용
# ---------------------------------------------------

prompt = ChatPromptTemplate.from_template("{country}의 수도는 어디야?").configurable_alternatives(
    ConfigurableField(id="prompt"),
    default_key="capital",

    area=ChatPromptTemplate.from_template("{country}의 면적은 얼마야?"),
    population=ChatPromptTemplate.from_template("{country}의 인구수는 얼마야?")
)

chain = prompt | llm


In [15]:
# 기본 프롬프트로 실행
res = chain.invoke({"country": "북한"})
res

AIMessage(content='북한의 수도는 평양입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 15, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5714259854', 'id': 'chatcmpl-DOyeVzPv5b9KiWnb8kPwjSm77S8OT', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019d3d07-70f9-7220-9271-35c79f647d3d-0', usage_metadata={'input_tokens': 15, 'output_tokens': 9, 'total_tokens': 24, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [ ]:
# 다른 프롬프트로 실행
res = chain.with_config(configurable={"prompt": "area"}).invoke({"country": "북한"})
res

AIMessage(content='북한의 면적은 약 120,540 평방킬로미터입니다. 이는 한반도의 북부에 위치한 지역으로, 남한보다 약간 더 넓은 면적을 가지고 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 16, 'total_tokens': 62, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5714259854', 'id': 'chatcmpl-DOyfTbkUMeYy3ZaYDpDk34ixxioLO', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019d3d08-5c05-7410-966d-72eb8fd06d5a-0', usage_metadata={'input_tokens': 16, 'output_tokens': 46, 'total_tokens': 62, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [17]:
# 인구 프롬프트로 실행
res = chain.with_config(configurable={"prompt": "population"}).invoke({"country": "북한"})
res

AIMessage(content='2023년 기준으로 북한의 인구수는 약 2,500만 명 정도로 추정됩니다. 하지만 정확한 인구수는 다양한 요인에 따라 변동할 수 있으며, 북한 정부의 통계 발표에 따라 다를 수 있습니다. 최신 정보는 공식적인 통계나 국제기구의 자료를 참고하는 것이 좋습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 17, 'total_tokens': 92, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5714259854', 'id': 'chatcmpl-DOygTfniIyUjAsGPL7c07UNmhalfz', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019d3d09-4ec9-7fc3-9497-299dac6f11a8-0', usage_metadata={'input_tokens': 17, 'output_tokens': 75, 'total_tokens': 92, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## 4. 프롬프트와 LLM 모두 변경하기

프롬프트와 LLM을 모두 `configurable_alternatives`로 설정하면, 런타임에 둘 다 동시에 변경할 수 있어요.

In [ ]:
# ============================================================
# TODO: 프롬프트와 LLM 모두 configurable_alternatives로 설정하고 동시에 교체하세요
# 힌트: with_config(configurable={"prompt": "founder", "llm": "openai"})
# 예상 결과: 애플 창립자 질문을 gpt-4o로 답변
# ============================================================

# ---------------------------------------------------
# 프롬프트 + LLM 동시 configurable_alternatives 설정
# ---------------------------------------------------


In [ ]:
# 프롬프트와 LLM 모두 변경하여 실행


## 5. 설정 저장 및 재사용

특정 설정으로 구성된 체인을 별도의 변수에 저장하여 재사용할 수 있어요.

In [ ]:
# 특정 설정으로 체인 저장


## 마무리 요약

### configurable_fields vs configurable_alternatives 비교

| 항목 | `configurable_fields` | `configurable_alternatives` |
|------|-----------------------|-----------------------------|
| 변경 대상 | 객체의 파라미터 값 | Runnable 객체 자체 |
| 사용 예시 | model_name, temperature | 모델 교체, 프롬프트 교체 |
| 설정 방법 | `ConfigurableField(id=...)` | `ConfigurableField(id=...) + 키워드 인자` |
| 기본값 지정 | 객체 초기화 시 파라미터 | `default_key` |

### 설정 적용 방법

```python
# 1. invoke() 시 매번 전달
chain.invoke(input, config={"configurable": {"키": "값"}})

# 2. with_config()로 미리 고정
chain_v2 = chain.with_config(configurable={"키": "값"})
chain_v2.invoke(input)
```

### 다음 노트북 예고

다음 노트북(`07-RunnableWithMessageHistory.ipynb`)에서는 체인에 대화 기록을 추가하여 이전 대화 맥락을 기억하는 `RunnableWithMessageHistory` 사용법을 배워요. 세션별 대화 기록을 관리하는 챗봇 패턴을 살펴볼게요.